In [ ]:
%cd /content/ComfyUI

%env CUDA_VISIBLE_DEVICES=1

import os, shutil, json, requests, random, time
from urllib.parse import urlsplit

import torch
from PIL import Image
import numpy as np

import nodes
from nodes import NODE_CLASS_MAPPINGS
from nodes import load_custom_node
from comfy_extras import nodes_mochi

import asyncio
import execution
import server
loop = asyncio.new_event_loop()
asyncio.set_event_loop(loop)
server_instance = server.PromptServer(loop)
execution.PromptQueue(server)

load_custom_node("/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite")

CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
EmptyMochiLatentVideo = nodes_mochi.NODE_CLASS_MAPPINGS["EmptyMochiLatentVideo"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
VHS_VideoCombine = NODE_CLASS_MAPPINGS["VHS_VideoCombine"]()

with torch.inference_mode():
    clip = CLIPLoader.load_clip('t5xxl_fp16.safetensors', type="mochi")[0]
    model = UNETLoader.load_unet("mochi/mochi_preview_bf16.safetensors", "default")[0]
    vae = VAELoader.load_vae("mochi/mochi_preview_vae_decoder_bf16.safetensors")[0]

def closestNumber(n, m):
    q = int(n / m)
    n1 = m * q
    if (n * m) > 0:
        n2 = m * (q + 1)
    else:
        n2 = m * (q - 1)
    if abs(n - n1) < abs(n - n2):
        return n1
    return n2

@torch.inference_mode()
def generate(input):
    values = input["input"]

    positive_prompt = values['positive_prompt']
    negative_prompt = values['negative_prompt']
    seed = values['seed']
    steps = values['steps']
    cfg = values['cfg']
    sampler_name = values['sampler_name']
    scheduler = values['scheduler']
    width = values['width']
    height = values['height']
    length = values['length']
    
    if seed == 0:
        random.seed(int(time.time()))
        seed = random.randint(0, 18446744073709551615)
    print(seed)

    positive = CLIPTextEncode.encode(clip, positive_prompt)[0]
    negative = CLIPTextEncode.encode(clip, negative_prompt)[0]
    latent_image = EmptyMochiLatentVideo.generate(width, height, length, batch_size=1)[0]
    samples = KSampler.sample(model, seed, steps, cfg, sampler_name, scheduler, positive, negative, latent_image, denoise=1.0)[0]
    frames  = VAEDecode.decode(vae, samples)[0]
    out_video = VHS_VideoCombine.combine_video(images=frames, frame_rate=24, loop_count=0, filename_prefix="Mochi", format="video/h264-mp4", save_output=True, prompt=None, unique_id=None)
    source = out_video["result"][0][1][1]
    destination = '/content/ComfyUI/output/mochi-1-preview-tost.mp4'
    shutil.move(source, destination)
    result = '/content/ComfyUI/output/mochi-1-preview-tost.mp4'
    return result

In [ ]:
input = { 
    "input": {
        "positive_prompt": "Create a short, looped animation of a small, fluffy creature with big, round eyes and soft fur, similar to a baby Pikachu, emulating the adorable curiosity of baby animals. The creature should be sitting in a human hand, surrounded by a soft, natural forest setting with tiny flowers and gentle lighting. Subtle, lifelike movements include slight head tilts, gentle blinking, and small, curious paw or ear twitches. Its large eyes should glimmer with curiosity and warmth, occasionally widening as it looks around its environment, embodying the innocent curiosity and playfulness of a young animal. Keep the lighting soft, creating a warm, inviting atmosphere with faint sunlight filtering through, highlighting the softness of its fur.",
        "negative_prompt": "blurry",
        "seed": 18446744073709551615,
        "steps": 50,
        "cfg": 4.5,
        "sampler_name": "euler",
        "scheduler": "simple",
        "width": 848,
        "height": 480,
        "length": 37
    }
}
image = generate(input)
from IPython.display import Video
Video(image, embed=True)

In [ ]:
%cd /content/FasterCache/scripts/mochi

%env CUDA_VISIBLE_DEVICES=1

import sys
sys.path.append('/content/FasterCache/fastercache/models')

from genmo.mochi_preview.pipelines import (
    DecoderModelFactory,
    DitModelFactory,
    MochiSingleGPUPipeline,
    T5ModelFactory,
    linear_quadratic_schedule,
)
from genmo.lib.progress import progress_bar
from genmo.lib.utils import save_video

pipeline = MochiSingleGPUPipeline(text_encoder_factory=T5ModelFactory(),
                                 dit_factory=DitModelFactory(model_path=f"/content/ComfyUI/models/unet/mochi/dit.safetensors", model_dtype="bf16"),
                                 decoder_factory=DecoderModelFactory(model_path=f"/content/ComfyUI/models/vae/mochi/decoder.safetensors",
                                 model_stats_path=f"/content/ComfyUI/models/vae/mochi/vae_stats.json"),
                                 cpu_offload=True)

In [ ]:
import numpy as np
import torch

num_inference_steps = 64
cfg_scale = 4.5

# sigma_schedule should be a list of floats of length (num_inference_steps + 1),
# such that sigma_schedule[0] == 1.0 and sigma_schedule[-1] == 0.0 and monotonically decreasing.
sigma_schedule = linear_quadratic_schedule(num_inference_steps, 0.025, linear_steps=5)

# cfg_schedule should be a list of floats of length num_inference_steps.
# For simplicity, we just use the same cfg scale at all timesteps,
# but more optimal schedules may use varying cfg, e.g:
# [5.0] * (num_inference_steps // 2) + [4.5] * (num_inference_steps // 2)
cfg_schedule = [cfg_scale] * num_inference_steps

args = {
        "height": 480,
        "width": 848,
        "num_frames": 37,
        "sigma_schedule": sigma_schedule,
        "cfg_schedule": cfg_schedule,
        "num_inference_steps": num_inference_steps,
        # We *need* flash attention to batch cfg
        # and it's only worth doing in a high-memory regime (assume multiple GPUs)
        "batch_cfg": False,
        "prompt": "Create a short, looped animation of a small, fluffy creature with big, round eyes and soft fur, similar to a baby Pikachu, emulating the adorable curiosity of baby animals. The creature should be sitting in a human hand, surrounded by a soft, natural forest setting with tiny flowers and gentle lighting. Subtle, lifelike movements include slight head tilts, gentle blinking, and small, curious paw or ear twitches. Its large eyes should glimmer with curiosity and warmth, occasionally widening as it looks around its environment, embodying the innocent curiosity and playfulness of a young animal. Keep the lighting soft, creating a warm, inviting atmosphere with faint sunlight filtering through, highlighting the softness of its fur.",
        "negative_prompt": "",
        "seed": 12345,
        }

final_frames = pipeline(**args)
final_frames = final_frames[0]

In [3]:
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ShmuelRonen/ComfyUI-FreeMemory

/content/ComfyUI/custom_nodes
Cloning into 'ComfyUI-FreeMemory'...


/content/ComfyUI-venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 46 (delta 22), reused 7 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 19.27 KiB | 2.14 MiB/s, done.
Resolving deltas: 100% (22/22), done.
